In [1]:
# 09_raw_polygon_pointnet_wj_distill.ipynb
# Raw polygon/linestring encoder experiment, 10k first.
#
# Goal:
#   Test whether we can replace quadtree input to the MLP with raw geometry input.
#
# Important:
#   The GT is WeightedJaccard over quadtree vectors, so the raw encoder is trained
#   by distilling from the quadtree/WJ teacher and the existing MLP embedding model.


In [2]:
import os
import re
import pickle
import random
import time

import nmslib
import numpy as np
import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

THREADS = 32
QUERY_START_10K = 8000
PARKS_TSV = "/raid/ruban/data/parks.tsv"

class QuadtreeCompressorV1(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )
    def forward(self, x):
        return self.net(x)

class RawPointNetEncoder(nn.Module):
    def __init__(self, in_channels=4, emb_dim=512):
        super().__init__()
        self.point_mlp = nn.Sequential(
            nn.Conv1d(in_channels, 64, 1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 128, 1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 256, 1), nn.BatchNorm1d(256), nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.Linear(512, 512), nn.BatchNorm1d(512), nn.ReLU(),
            nn.Linear(512, emb_dim), nn.BatchNorm1d(emb_dim),
        )
    def forward(self, x):
        # x: (B, N, C)
        h = self.point_mlp(x.transpose(1, 2))
        h_max = h.max(dim=2).values
        h_mean = h.mean(dim=2)
        return self.head(torch.cat([h_max, h_mean], dim=1))

def get_mem_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 1024**2

def recall_at_k(gt_lookup, nbrs, query_start_id, k):
    total = 0.0
    count = 0
    for i, (ids, _) in enumerate(nbrs):
        gt = set(gt_lookup.get(query_start_id + i, [])[:k])
        if not gt:
            continue
        total += len(gt & set(ids[:k])) / len(gt)
        count += 1
    return total / count if count else 0.0

def eval_recall(gt_lookup, nbrs, query_start_id, max_k):
    return {k: recall_at_k(gt_lookup, nbrs, query_start_id, k)
            for k in (10, 50, 100, 500) if k <= max_k}

coord_pat = re.compile(r"[-+]?\d*\.\d+|[-+]?\d+")

def parse_wkt_coords(wkt):
    vals = [float(v) for v in coord_pat.findall(wkt)]
    if len(vals) < 4:
        return None
    # Some WKT rows may contain an odd extra numeric token from malformed text or
    # dimensional markers. Keep complete x/y pairs rather than failing the run.
    if len(vals) % 2 == 1:
        vals = vals[:-1]
    arr = np.asarray(vals, dtype=np.float32).reshape(-1, 2)
    arr = arr[np.isfinite(arr).all(axis=1)]
    # Parks data is lon/lat; this removes extreme parser artifacts while keeping
    # valid UK coordinates.
    plausible = ((arr[:, 0] >= -180) & (arr[:, 0] <= 180) &
                 (arr[:, 1] >= -90) & (arr[:, 1] <= 90))
    arr = arr[plausible]
    return arr if len(arr) >= 2 else None

def load_raw_coords_tsv(path, n_rows):
    coords = []
    ids = []
    with open(path, "r", errors="replace") as f:
        for i, line in enumerate(tqdm(f, total=n_rows, desc="Reading TSV")):
            if i >= n_rows:
                break
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 2:
                coords.append(None); ids.append(None); continue
            ids.append(parts[0])
            coords.append(parse_wkt_coords(parts[1]))
    return ids, coords

def sample_polyline(coords, n_points):
    if coords is None or len(coords) < 2:
        return np.zeros((n_points, 2), dtype=np.float32)
    seg = coords[1:] - coords[:-1]
    lens = np.sqrt((seg * seg).sum(axis=1))
    total = float(lens.sum())
    if total <= 1e-12:
        return np.repeat(coords[:1], n_points, axis=0).astype(np.float32)
    cum = np.concatenate([[0.0], np.cumsum(lens)])
    targets = np.linspace(0.0, total, n_points, endpoint=True)
    out = np.empty((n_points, 2), dtype=np.float32)
    j = 0
    for ti, t in enumerate(targets):
        while j < len(lens) - 1 and cum[j + 1] < t:
            j += 1
        alpha = 0.0 if lens[j] <= 1e-12 else (t - cum[j]) / lens[j]
        out[ti] = coords[j] * (1 - alpha) + coords[j + 1] * alpha
    return out

def build_point_tensor(coords_list, n_points=128):
    sampled = np.stack([sample_polyline(c, n_points) for c in coords_list]).astype(np.float32)
    # Global normalized coordinates preserve location, which matters for quadtree WJ.
    valid = sampled.reshape(-1, 2)
    mn = valid.min(axis=0)
    mx = valid.max(axis=0)
    scale = np.maximum(mx - mn, 1e-6)
    global_xy = (sampled - mn) / scale
    # Local normalized shape coordinates add geometry/shape signal.
    center = sampled.mean(axis=1, keepdims=True)
    local = sampled - center
    local_scale = np.maximum(np.sqrt((local * local).sum(axis=2)).max(axis=1, keepdims=True), 1e-6)
    local_xy = local / local_scale[:, None, :]
    points = np.concatenate([global_xy, local_xy], axis=2).astype(np.float32)
    return points, {"global_min": mn, "global_scale": scale}


In [3]:
# Configuration, 10k first
n_total = 10_000
query_start = QUERY_START_10K
device = torch.device("cuda:0")
seed = 123

n_points = 128
teacher_variant = "harddist"  # use hard-negative MLP teacher if available, else base
hard_pool_k = 500
exclude_gt_top = 500
positive_per_query = 5
hard_neg_per_positive = 2

batch_size = 128
epochs = 20
lr = 1e-3
weight_decay = 1e-4
margin = 0.05
rank_weight = 1.0
teacher_weight = 0.50

candidate_ks = [100, 200, 500]
rerank_batch_size = 16
out_path = "/tmp/results_raw_pointnet_wjdistill.pkl"
raw_ckpt = "/tmp/best_raw_pointnet_wjdistill_10k.pt"
points_cache = "/tmp/raw_points_10k_128.npy"

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)


In [4]:
# Load quadtree vectors, WJ ground truth, and raw coordinate samples.
qt = np.load("/tmp/qt_10k.npy")[:n_total]
with open("/tmp/gt_lookup_10k.pkl", "rb") as f:
    gt = pickle.load(f)
corpus_qt = qt[:query_start]
query_qt = qt[query_start:]
corpus_sums = corpus_qt.sum(axis=1)

if os.path.exists(points_cache):
    points = np.load(points_cache)
    print(f"Loaded cached points: {points.shape}")
else:
    raw_ids, coords_list = load_raw_coords_tsv(PARKS_TSV, n_total)
    points, norm_info = build_point_tensor(coords_list, n_points=n_points)
    np.save(points_cache, points)
    print(f"Saved sampled points: {points_cache}")

print(f"qt={qt.shape} | points={points.shape}")
print(f"corpus points={points[:query_start].shape} | query points={points[query_start:].shape}")


Reading TSV: 100%|██████████| 10000/10000 [00:00<00:00, 22360.37it/s]


Saved sampled points: /tmp/raw_points_10k_128.npy
qt=(10000, 18499) | points=(10000, 128, 4)
corpus points=(8000, 128, 4) | query points=(2000, 128, 4)


In [5]:
# Build teacher MLP embeddings and mine hard negatives from teacher cosine index.
base_ckpt = "/tmp/best_compressor_v1_clean.pt"
hard_ckpt = "/tmp/best_compressor_hardneg_wjdistill_10k.pt"
teacher_ckpt = hard_ckpt if teacher_variant == "harddist" and os.path.exists(hard_ckpt) else base_ckpt

teacher_mlp = QuadtreeCompressorV1(qt.shape[1], out_dim=512).to(device)
teacher_mlp.load_state_dict(torch.load(teacher_ckpt, weights_only=True, map_location=device))
teacher_mlp.eval()

teacher_embs = []
with torch.no_grad():
    for start in tqdm(range(0, len(qt), 512), desc="Teacher MLP embeddings"):
        batch = torch.tensor(qt[start:start + 512], dtype=torch.float32, device=device)
        teacher_embs.append(F.normalize(teacher_mlp(batch), dim=1).cpu().numpy())
teacher_embs = np.vstack(teacher_embs).astype(np.float32)

idx = nmslib.init(method="hnsw", space="cosinesimil")
for i in tqdm(range(query_start), desc="Adding teacher corpus"):
    idx.addDataPoint(i, teacher_embs[i])
idx.createIndex({"M": 20, "efConstruction": 200, "post": 1}, print_progress=True)
idx.setQueryTimeParams({"efSearch": 200})

query_ids = [qid for qid in sorted(gt) if query_start <= qid < n_total]
local_q = [qid - query_start for qid in query_ids]
print(f"Mining hard negatives for {len(query_ids)} queries")
nbrs = idx.knnQueryBatch(teacher_embs[query_start:][local_q], k=hard_pool_k, num_threads=THREADS)


Adding teacher corpus: 100%|██████████| 8000/8000 [00:00<00:00, 656231.56it/s]

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

Mining hard negatives for 1818 queries


In [6]:
# Build hard-negative triplets for raw encoder training.
triplets = []
for qid, (cand_ids, _) in tqdm(zip(query_ids, nbrs), total=len(query_ids), desc="Triplets"):
    positives = [pid for pid in gt.get(qid, []) if 0 <= pid < query_start]
    if not positives:
        continue
    pos_train = positives[:positive_per_query]
    exclude = set(positives[:exclude_gt_top])
    hard_negs = [int(cid) for cid in cand_ids if int(cid) not in exclude]
    if not hard_negs:
        continue
    for pos_id in pos_train:
        for j in range(hard_neg_per_positive):
            triplets.append((qid, pos_id, hard_negs[min(j, len(hard_negs) - 1)]))

random.shuffle(triplets)
val_n = max(1, int(0.1 * len(triplets)))
val_triplets = triplets[:val_n]
train_triplets = triplets[val_n:]
print(f"triplets={len(triplets):,} | train={len(train_triplets):,} | val={len(val_triplets):,}")
print("sample", triplets[0] if triplets else None)


Triplets: 100%|██████████| 1818/1818 [00:00<00:00, 6462.52it/s]

triplets=17,440 | train=15,696 | val=1,744
sample (9234, 5871, 2969)


In [7]:
class RawTripletDataset(Dataset):
    def __init__(self, points, teacher_embs, triplets):
        self.points = points
        self.teacher_embs = teacher_embs
        self.triplets = triplets
    def __len__(self):
        return len(self.triplets)
    def __getitem__(self, idx):
        qid, pos_id, neg_id = self.triplets[idx]
        return (
            torch.from_numpy(self.points[qid]).float(),
            torch.from_numpy(self.points[pos_id]).float(),
            torch.from_numpy(self.points[neg_id]).float(),
            torch.from_numpy(self.teacher_embs[qid]).float(),
            torch.from_numpy(self.teacher_embs[pos_id]).float(),
            torch.from_numpy(self.teacher_embs[neg_id]).float(),
        )

train_loader = DataLoader(RawTripletDataset(points, teacher_embs, train_triplets),
                          batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(RawTripletDataset(points, teacher_embs, val_triplets),
                        batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

def raw_loss(model, q, p, n, tq, tp, tn):
    zq = F.normalize(model(q), dim=1)
    zp = F.normalize(model(p), dim=1)
    zn = F.normalize(model(n), dim=1)
    sim_pos = F.cosine_similarity(zq, zp)
    sim_neg = F.cosine_similarity(zq, zn)
    rank = F.relu(margin - sim_pos + sim_neg).mean()
    teacher = (F.mse_loss(zq, tq) + F.mse_loss(zp, tp) + F.mse_loss(zn, tn)) / 3.0
    return rank_weight * rank + teacher_weight * teacher, rank.detach(), teacher.detach(), sim_pos.detach().mean(), sim_neg.detach().mean()


In [8]:
# Train raw PointNet encoder.
raw_model = RawPointNetEncoder(in_channels=points.shape[2], emb_dim=512).to(device)
optimizer = torch.optim.AdamW(raw_model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
best_val = float("inf")
history = []

for epoch in range(1, epochs + 1):
    raw_model.train()
    train_losses = []
    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{epochs} train", leave=False)
    for q, p, n, tq, tp, tn in pbar:
        q = q.to(device, non_blocking=True); p = p.to(device, non_blocking=True); n = n.to(device, non_blocking=True)
        tq = tq.to(device, non_blocking=True); tp = tp.to(device, non_blocking=True); tn = tn.to(device, non_blocking=True)
        loss, rank, teacher, sim_pos, sim_neg = raw_loss(raw_model, q, p, n, tq, tp, tn)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(raw_model.parameters(), 1.0)
        optimizer.step()
        train_losses.append(float(loss.detach().cpu()))
        pbar.set_postfix(loss=f"{train_losses[-1]:.4f}", rank=f"{float(rank):.4f}", teacher=f"{float(teacher):.4f}")

    raw_model.eval()
    val_losses = []
    with torch.no_grad():
        for q, p, n, tq, tp, tn in val_loader:
            q = q.to(device, non_blocking=True); p = p.to(device, non_blocking=True); n = n.to(device, non_blocking=True)
            tq = tq.to(device, non_blocking=True); tp = tp.to(device, non_blocking=True); tn = tn.to(device, non_blocking=True)
            loss, rank, teacher, sim_pos, sim_neg = raw_loss(raw_model, q, p, n, tq, tp, tn)
            val_losses.append(float(loss.detach().cpu()))
    train_loss = float(np.mean(train_losses))
    val_loss = float(np.mean(val_losses))
    scheduler.step()
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
    if val_loss < best_val:
        best_val = val_loss
        torch.save(raw_model.state_dict(), raw_ckpt)
    print(f"Epoch {epoch:02d} | train={train_loss:.5f} | val={val_loss:.5f} | best={best_val:.5f}")

print(f"Done. Best raw checkpoint: {raw_ckpt}")


Epoch 01 | train=0.12420 | val=0.11011 | best=0.11011


Epoch 02 | train=0.09137 | val=0.09131 | best=0.09131


Epoch 03 | train=0.06769 | val=0.07347 | best=0.07347


Epoch 04 | train=0.05150 | val=0.05849 | best=0.05849


Epoch 05 | train=0.03767 | val=0.04791 | best=0.04791


...

Epoch 16 | train=0.00232 | val=0.00777 | best=0.00777


Epoch 17 | train=0.00226 | val=0.00720 | best=0.00720


Epoch 18 | train=0.00212 | val=0.00685 | best=0.00685


Epoch 19 | train=0.00209 | val=0.00729 | best=0.00685


Epoch 20 | train=0.00205 | val=0.00686 | best=0.00685
Done. Best raw checkpoint: /tmp/best_raw_pointnet_wjdistill_10k.pt


In [9]:
# Evaluate raw encoder as candidate generator, with exact WJ GPU rerank using quadtree teacher metric.
def generate_raw_embeddings(model, points, device, batch_size=512):
    model.eval()
    chunks = []
    with torch.no_grad():
        for start in tqdm(range(0, len(points), batch_size), desc="Raw embeddings"):
            batch = torch.tensor(points[start:start + batch_size], dtype=torch.float32, device=device)
            chunks.append(F.normalize(model(batch), dim=1).cpu().numpy())
    return np.vstack(chunks).astype(np.float32)

def rerank_wj_gpu(query_qt, nbrs_raw, corpus_qt, corpus_sums, device, batch_size=16):
    corpus_t = torch.from_numpy(corpus_qt).to(device=device, dtype=torch.float32)
    corpus_sums_t = torch.from_numpy(corpus_sums).to(device=device, dtype=torch.float32)
    reranked = [None] * len(nbrs_raw)
    for start in tqdm(range(0, len(nbrs_raw), batch_size), desc="GPU WJ rerank"):
        batch = nbrs_raw[start:start + batch_size]
        groups = {}
        for offset, (ids, _) in enumerate(batch):
            ids_arr = np.asarray(ids, dtype=np.int64)
            groups.setdefault(len(ids_arr), []).append((start + offset, ids_arr))
        for _, items in groups.items():
            ids_np = np.stack([ids for _, ids in items], axis=0)
            query_np = np.stack([query_qt[i] for i, _ in items], axis=0)
            ids_t = torch.from_numpy(ids_np).to(device=device)
            q_t = torch.from_numpy(query_np).to(device=device, dtype=torch.float32)
            c_t = corpus_t[ids_t]
            mins = torch.minimum(q_t[:, None, :], c_t).sum(dim=2)
            maxs = q_t.sum(dim=1, keepdim=True) + corpus_sums_t[ids_t] - mins
            order = torch.argsort(mins / maxs.clamp_min(1e-10), dim=1, descending=True).cpu().numpy()
            for row, (absolute_i, ids) in zip(order, items):
                reranked[absolute_i] = (ids[row].tolist(), [])
    del corpus_t, corpus_sums_t
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return reranked

def evaluate_raw(model, label):
    print("\n" + "=" * 80)
    print(label)
    print("=" * 80)
    embs = generate_raw_embeddings(model, points, device)
    corpus_embs = embs[:query_start]
    query_embs = embs[query_start:]
    idx = nmslib.init(method="hnsw", space="cosinesimil")
    for i in tqdm(range(len(corpus_embs)), desc="Adding raw corpus"):
        idx.addDataPoint(i, corpus_embs[i])
    t0 = time.time()
    idx.createIndex({"M": 20, "efConstruction": 200, "post": 1}, print_progress=True)
    build_s = time.time() - t0
    idx.setQueryTimeParams({"efSearch": 200})
    results = {}
    for k in candidate_ks:
        t0 = time.time()
        nbrs = idx.knnQueryBatch(query_embs, k=k, num_threads=THREADS)
        hnsw_s = time.time() - t0
        rec_no = eval_recall(gt, nbrs, query_start, max_k=k)
        t0 = time.time()
        nbrs_rr = rerank_wj_gpu(query_qt, nbrs, corpus_qt, corpus_sums, device, rerank_batch_size)
        rerank_s = time.time() - t0
        rec_rr = eval_recall(gt, nbrs_rr, query_start, max_k=k)
        qps = len(query_embs) / (hnsw_s + rerank_s)
        results[f"k{k}"] = {"no_rerank": rec_no, "wj_rerank": rec_rr, "qps": qps,
                             "hnsw_s": hnsw_s, "rerank_s": rerank_s, "build_s": build_s}
        print(f"K={k} | QPS={qps:.1f} | no-rerank R@100={rec_no.get(100, float('nan')):.4f} | WJ-rerank R@100={rec_rr.get(100, float('nan')):.4f}")
        for kk, rr in rec_rr.items():
            print(f"  rerank R@{kk:<4} = {rr:.4f}")
    return results

best_raw = RawPointNetEncoder(in_channels=points.shape[2], emb_dim=512).to(device)
best_raw.load_state_dict(torch.load(raw_ckpt, weights_only=True, map_location=device))
raw_results = evaluate_raw(best_raw, "Raw PointNet + cosine candidates + exact WJ rerank")



Raw PointNet + cosine candidates + exact WJ rerank


Adding raw corpus: 100%|██████████| 8000/8000 [00:00<00:00, 703801.33it/s]

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
GPU WJ rerank: 100%|██████████| 125/125 [00:00<00:00, 623.39it/s]


K=100 | QPS=5908.8 | no-rerank R@100=0.0137 | WJ-rerank R@100=0.0137
  rerank R@10   = 0.0141
  rerank R@50   = 0.0137
  rerank R@100  = 0.0137


GPU WJ rerank: 100%|██████████| 125/125 [00:00<00:00, 509.60it/s]


K=200 | QPS=4318.3 | no-rerank R@100=0.0137 | WJ-rerank R@100=0.0288
  rerank R@10   = 0.0316
  rerank R@50   = 0.0290
  rerank R@100  = 0.0288


GPU WJ rerank: 100%|██████████| 125/125 [00:00<00:00, 293.77it/s]


K=500 | QPS=3227.1 | no-rerank R@100=0.0137 | WJ-rerank R@100=0.0719
  rerank R@10   = 0.0830
  rerank R@50   = 0.0732
  rerank R@100  = 0.0719
  rerank R@500  = 0.0711


In [10]:
# Save results.
run_key = time.strftime("10k_raw_pointnet_%Y%m%d_%H%M%S")
record = {
    "config": {
        "n_points": n_points,
        "teacher_ckpt": teacher_ckpt,
        "hard_pool_k": hard_pool_k,
        "positive_per_query": positive_per_query,
        "hard_neg_per_positive": hard_neg_per_positive,
        "epochs": epochs,
        "lr": lr,
        "margin": margin,
        "teacher_weight": teacher_weight,
        "candidate_ks": candidate_ks,
    },
    "history": history,
    "raw_results": raw_results,
}
try:
    with open(out_path, "rb") as f:
        saved = pickle.load(f)
except FileNotFoundError:
    saved = {"runs": {}}
saved.setdefault("runs", {})[run_key] = record
with open(out_path, "wb") as f:
    pickle.dump(saved, f)
print(f"Saved {run_key} to {out_path}")


Saved 10k_raw_pointnet_20260427_164655 to /tmp/results_raw_pointnet_wjdistill.pkl
